## Intro

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.host", "localhost") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 12:04:20 WARN Utils: Your hostname, codespaces-ad5d43, resolves to a loopback address: 127.0.0.1; using 10.0.11.130 instead (on interface eth0)
26/03/08 12:04:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 12:04:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Zones

In [28]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 12:21:54--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.147, 18.239.38.83, 18.239.38.181, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.147|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 12:21:54 (207 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [30]:
!head data/taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


In [31]:
zones_df = spark.read \
    .option("header", "true") \
    .csv('data/taxi_zone_lookup.csv')

In [33]:
zones_df.show(n = 5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [35]:
zones_df.write.parquet('data/zones')

In [37]:
!ls data -lh

total 24K
drwxr-xr-x+ 3 codespace codespace 4.0K Mar  8 11:51 report
-rw-rw-rw-  1 codespace codespace  13K Feb 22  2024 taxi_zone_lookup.csv
drwxr-xr-x+ 2 codespace codespace 4.0K Mar  8 12:23 zones


## Green & Yellow

In [3]:
df_green = (spark.read
  .option("recursiveFileLookup", "true")
  .parquet("file:/workspaces/data-engineering-zoomcamp/06-batch/code/data/pq/green/"))

In [4]:
df_green.createOrReplaceTempView ('green')

In [5]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [9]:
df_green_revenue.show(n=5)

[Stage 2:=======================================>                   (2 + 1) / 3]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-28 19:00:00| 134|193.61000000000007|            17|
|2020-01-22 19:00:00|  65| 657.0300000000001|            41|
|2020-01-27 08:00:00|  17|             85.56|             4|
|2020-01-02 09:00:00|  66|229.39999999999998|            12|
|2020-01-02 12:00:00|  89|310.28000000000003|            14|
+-------------------+----+------------------+--------------+
only showing top 5 rows


In [6]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

## GroupBy

In [7]:
df_yellow = (spark.read
  .option("recursiveFileLookup", "true")
  .parquet("file:/workspaces/data-engineering-zoomcamp/06-batch/code/data/pq/yellow/"))

In [8]:
df_yellow.createOrReplaceTempView ('yellow')

In [9]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [10]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [13]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

## Joins

In [14]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [19]:
df_join = df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [15]:
df_join.show()

[Stage 21:>                                                         (0 + 1) / 1]

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|   7| 769.7299999999996|                  45| 769.7299999999996|                   45|
|2020-01-01 00:00:00|  18|               7.8|                   1|               7.8|                    1|
|2020-01-01 00:00:00|  29|              61.3|                   1|              61.3|                    1|
|2020-01-01 00:00:00|  36|295.34000000000003|                  11|295.34000000000003|                   11|
|2020-01-01 00:00:00|  37|            175.67|                   6|            175.67|                    6|
|2020-01-01 00:00:00|  38| 98.78999999999999|                   2| 98.78999999999999|                    2|
|2020-01-01 00:00:00|  40|16

In [16]:
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [17]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')

In [21]:
df_join = spark.read.parquet('data/report/revenue/total')

In [22]:
df_join.show(n=5)

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  17|195.03000000000003|                   9|195.03000000000003|                    9|
|2020-01-01 00:00:00|  22|              15.8|                   1|              15.8|                    1|
|2020-01-01 00:00:00|  24|              87.6|                   3|              87.6|                    3|
|2020-01-01 00:00:00|  25| 531.0000000000002|                  26| 531.0000000000002|                   26|
|2020-01-01 00:00:00|  32| 68.94999999999999|                   2| 68.94999999999999|                    2|
+-------------------+----+------------------+--------------------+------------------+---------------------+
only showing top 5 rows


In [38]:
df_zones = spark.read.parquet('data/zones/')

In [39]:
df_zones.show(n=5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [41]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [45]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

In [47]:
spark.stop()